# Orbit Wars — Colab Training Pipeline (v2/v4 OrbitNet, A100)

GPU-accelerated training on Colab with Google Drive persistence.

This notebook runs the **v2 OrbitNet** with the four structural fixes and the
**Expert Iteration (ExIt)** pivot from `rl_research/REPORT.md`, plus the **v4
"ceiling"** recipe (`PIPELINE='v4'`, the default) from `rl_research/V3_STALL_PLAYBOOK.md`:

- **Fix 1 — dedicated masked ship-fraction head** (factored `(target, fraction)`;
  BC clones fleet sizes; PPO/ExIt get a real gradient on quantity).
- **Fix 2 — mean per-planet entropy** (stable exploration pressure).
- **Fix 3 — symlog value targets** (`ppo.value_symlog`, scale-robust value).
- **Fix 4 — longer rollouts** (episodes complete -> terminal signal reaches updates).

Pipelines (set `PIPELINE` in the config cell):
1. **`v4`** (default) — the full ceiling: PBRS reward + PFSP opponent pool + persistent
   BC anchor + **PopArt** + **PPG aux value head** + **shot validator** + rich
   representation (pair/comet/timeline features). Submits via `agent_v3.py` (config-driven).
2. **`v3`** — v4's predecessor: PPO + PBRS + PFSP + Tier-1 pair/comet features.
3. **`exit`** — BC-clone apex -> per-planet lookahead **search against the ground-truth
   simulator** -> supervised distillation. Planning, not policy-gradient.
4. **`ppo`** — v2 PPO with all four fixes + parallel CPU rollout collection.
5. **`exit_blend`** — **Phase 3 stronger-expert-search**: heuristic ExIt + a *grounded learned value at search leaves, blended* (`score=(1-w)·z(heur)+w·z(neural)`), warm-started from the iter-20 ckpt. The A/B over `w` is the current experiment (see the *Phase 3 gate* cell and `rl_research/STRONGER_EXPERT_SEARCH_PLAN.md`).
6. **`gumbel`** — **Phase 5 stronger-expert-search** (current): **Gumbel/Sequential-Halving candidate selection** anchored to the policy prior (Danihelka 2022), optionally with a **strong-net in-sim opponent** (`NET_OPPONENT`). The literature-backed replacement for the refuted value-blend + weak rollout opponent. Warm-started from the iter-20 ckpt; A/B vs the `GUMBEL_SEARCH=False` control (see the *Phase 5 gate* cell).

### A100 efficiency note
The OrbitNet is tiny (~0.6M params), so the **GPU is never the compute bottleneck** —
game simulation (collection) and lookahead (search) are **CPU-bound**. We therefore:
- enable **TF32** and use **large batches** so the A100 chews through BC / distillation /
  PPO updates,
- fan **search across all vCPUs** (`exit.search_workers`) and run **parallel rollout
  workers** (`ppo.num_workers`) for the PPO path,
- **cache apex demos on Drive** so the one-time collection isn't repeated across runs.

**Throughput note (v4):** rollout is CPU-bound on `encode_features` (~53% of step cost,
dominated by incoming-fleet prediction). **Phase A** vectorized that (~3.6x cheaper/step
on realistic states, bit-identical). **Phase B** made the parallel rollout path
v4-complete — PFSP + PopArt now run across subprocess workers — so v4 fans collection
across all vCPUs (`ppo.num_workers=CPU_COUNT`), keeping every v4 feature.

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# v2 needs only kaggle-environments + the usual scientific stack (no SB3).
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')
print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification + A100 performance flags

import os, torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # A100 / Ampere: enable TF32 for fast fp32 matmul (free speedup).
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = 'cuda'
else:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU (A100).')
    DEVICE = 'cpu'

CPU_COUNT = os.cpu_count() or 8
print(f'\nDevice: {DEVICE} | vCPUs: {CPU_COUNT}')
print('Note: collection/search are CPU-bound; the GPU accelerates neural training.')

## Configuration

Pick the pipeline and apply A100-tuned overrides. The base YAMLs
(`configs/v2_exit.yaml`, `configs/v2_bc.yaml`) already carry the four fixes;
here we scale batch sizes for the GPU and parallelism for the vCPUs, and point
all outputs (incl. the cached apex demo set) at Google Drive.

**Pipelines:** `ppo` / `v3` / `v4` (model-free), `exit` (heuristic ExIt — best so far, 77% vs apex), `exit_neural` (Phase 5 pure-neural swap, parked), `exit_rollout` (Phase 1, confirmed negative), and **`exit_blend`** (Phase 3 stronger-expert-search: **grounded learned value at search leaves, BLENDED with the heuristic** — `score=(1-w)·z(heur)+w·z(neural)`, warm-started from the iter-20 checkpoint — see `rl_research/STRONGER_EXPERT_SEARCH_PLAN.md`).

### Phase 3 A/B (`exit_blend`)
The blend weight `w` is the experiment. Run the pipeline **once per weight** in
`{0.0, 0.3, 0.5, 0.7}` by setting `VALUE_LEAF_BLEND` below and re-running Train —
each weight gets its **own** `run_name` (`v2_exit_blend_wNNN_a100`) so Drive
checkpoints never collide. `w=0.0` is the control (continue ExIt from iter-20 with
no blend). Then run the **Phase 3 gate** cell to score every weight with
`scripts/eval_fast.py` (side-alternated, paired seeds, multiple seed batches). A
weight ships only if it beats the `w=0` control on the **same** seeds.

### Phase 5 A/B (`gumbel`)
`gumbel` warm-starts from the iter-20 checkpoint and distills a Gumbel AlphaZero
*improved policy over the prior* (Build 1), optionally rolling out the strong net
as the in-sim opponent (Build 2, `NET_OPPONENT`). Run the **control** first
(`GUMBEL_SEARCH=False` -> `v2_exit_gumbel_ctrl_a100`), then the experiment
(`GUMBEL_SEARCH=True` -> `v2_exit_gumbel_on_a100`); each gets its own `run_name`
so Drive checkpoints never collide. Then run the **Phase 5 gate** cell — a variant
ships only if it beats the control on the **same** seeds. **Gate Build 1 alone
before enabling `NET_OPPONENT`** (`v2_exit_gumbel_netopp_a100`).


In [ ]:
#@title 3. Experiment Config

from v2.config import load_v2_config

PIPELINE = 'ppo'  #@param ['ppo', 'exit', 'gumbel', 'exit_blend', 'exit_rollout', 'exit_neural', 'v3', 'v4']
# Phase 3 (exit_blend only): leaf-value blend weight. Run once per value in
# {0.0, 0.3, 0.5, 0.7}; each gets its own run_name so the A/B never collides.
VALUE_LEAF_BLEND = 0.3  #@param {type:"number"}

# Phase 5 (gumbel only): Build 1 = Gumbel/Sequential-Halving candidate selection;
# Build 2 = strong-net in-sim opponent. Run the CONTROL once (GUMBEL_SEARCH=False)
# then the experiment(s) -- each variant gets its own run_name so the A/B never
# collides on Drive. Gate Build 1 alone before enabling NET_OPPONENT.
GUMBEL_SEARCH  = True   #@param {type:"boolean"}
NET_OPPONENT   = False  #@param {type:"boolean"}
GUMBEL_C_SCALE = 1.0    #@param {type:"number"}

# Exit-family pipelines all run v2.exit_train (search -> distill), not v2.train.
EXIT_FAMILY = ('exit', 'gumbel', 'exit_blend', 'exit_neural', 'exit_rollout')

# v3 = ppo recipe + PBRS reward + PFSP opponent pool + Tier-1 features
#      (pairwise travel-time/required-ships + comet targeting).
# v4 = v3 + persistent BC anchor + PopArt + PPG aux value head + shot validator
#      + rich representation (the full V3_STALL_PLAYBOOK ceiling).
_base = {'exit': 'configs/v2_exit.yaml', 'ppo': 'configs/v2_bc.yaml',
         'v3': 'configs/v3_features.yaml', 'v4': 'configs/v4_ceiling.yaml',
         'exit_neural': 'configs/v2_exit_neural.yaml',
         'exit_blend': 'configs/v2_exit_neural.yaml',
         'gumbel': 'configs/v2_exit_gumbel.yaml',
         'exit_rollout': 'configs/v2_exit_rollout.yaml'}[PIPELINE]
cfg = load_v2_config(_base)
WARM_START = None  # exit_* phases set this to the iter-20 warm-start checkpoint

# Common: device + Drive persistence + demo cache
cfg.device = 'cuda'
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'
cfg.imitation.bc_cache_path = f'{DRIVE_OUTPUT}/demos_apex_v2.pkl'  # v2 = higher-fidelity mapping; collect once, reuse

if PIPELINE in EXIT_FAMILY:
    cfg.run_name = 'v2_exit_a100'
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.imitation.bc_batch_size = 1024
    cfg.exit.iterations = 60
    cfg.exit.games_per_iter = 12
    cfg.exit.search_depth = 16
    cfg.exit.search_candidates = 12
    cfg.exit.train_epochs = 6
    cfg.exit.train_batch_size = 2048
    cfg.exit.dataset_max_iters = 4
    cfg.exit.search_workers = CPU_COUNT
    cfg.exit.collect_workers = CPU_COUNT   # parallel game collection (the ExIt bottleneck)
    cfg.exit.collect_fast_env = True       # engine-faithful standalone sim (no Kaggle harness)
    cfg.checkpoint_every = 5
    # Side-alternated, paired-seed periodic eval (the trustworthy gate, mirrors
    # scripts/eval_fast.py). eval_seed=20000 is shared so the in-training number
    # is directly comparable to the gate cell.
    cfg.eval.eval_every = 5
    cfg.eval.eval_games = 40
    cfg.eval.eval_seed = 20000
    cfg.eval.eval_workers = CPU_COUNT
else:  # ppo with all four fixes
    cfg.run_name = 'v2_ppo_a100'
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.ppo.num_workers = CPU_COUNT     # parallel CPU rollout collection (key lever)
    cfg.ppo.num_envs = 8
    cfg.ppo.rollout_steps = 128         # Fix 4: episodes complete
    cfg.ppo.total_updates = 5000
    cfg.ppo.minibatch_size = 2048       # large batch for A100 throughput
    cfg.ppo.epochs = 4
    cfg.ppo.gamma = 0.997               # long-horizon (meaningful with symlog)
    cfg.ppo.value_symlog = True         # Fix 3
    cfg.ppo.ent_coef = 0.01
    cfg.ppo.ent_coef_end = 0.0          # entropy anneal (Fix 2 is in the code)
    cfg.eval.eval_every = 250
    cfg.eval.eval_games = 20

if PIPELINE == 'v3':
    # v3 runs the PFSP+PBRS sequential loop, so it cannot use subprocess
    # workers (live opponent models). A100 still chews the big-batch updates.
    cfg.run_name = 'v3_a100'
    cfg.imitation.bc_cache_path = f'{DRIVE_OUTPUT}/demos_apex_v3.pkl'  # v3 feats -> own cache
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.imitation.bc_batch_size = 1024
    cfg.ppo.num_workers = 0             # PFSP requires sequential
    cfg.ppo.num_envs = 8
    cfg.ppo.rollout_steps = 96
    cfg.ppo.total_updates = 4000
    cfg.ppo.minibatch_size = 2048
    cfg.ppo.value_symlog = True
    cfg.ppo.ent_coef_end = 0.0
    cfg.eval.eval_every = 100
    cfg.eval.eval_games = 20

if PIPELINE == 'v4':
    # v4 "ceiling": the full V3_STALL_PLAYBOOK. The v4_ceiling.yaml already sets
    # PBRS reward, PFSP pool, PopArt, the PPG aux + shot-validator heads, the BC
    # anchor (coef_floor) and the rich representation -- so we only re-assert the
    # bits the generic 'ppo' block above stomps, and apply A100/Drive tuning.
    #
    # Throughput: Phase A vectorized encode_features (~3.6x cheaper/step) AND
    # Phase B made the parallel rollout path v4-complete -- PFSP + PopArt now work
    # across subprocess workers. So we fan rollout collection across all vCPUs;
    # each worker owns 1 env, so effective parallelism = num_workers (num_envs is
    # forced to 1 per worker) and samples/update = num_workers * rollout_steps.
    cfg.run_name = 'v4_ceiling'
    cfg.imitation.bc_cache_path = f'{DRIVE_OUTPUT}/demos_apex_v4.pkl'  # v4 feats -> own cache
    cfg.imitation.enabled = True
    cfg.imitation.bc_games = 300
    cfg.imitation.bc_epochs = 60
    cfg.imitation.bc_batch_size = 1024
    cfg.ppo.num_workers = CPU_COUNT     # Phase B: parallel CPU rollout (PFSP+PopArt across workers)
    cfg.ppo.num_envs = 1               # per-worker (workers each own 1 env)
    cfg.ppo.rollout_steps = 128
    cfg.ppo.total_updates = 5000
    cfg.ppo.minibatch_size = 2048
    cfg.ppo.value_symlog = False        # v4 uses PopArt, not symlog (mutually exclusive)
    cfg.ppo.popart = True
    cfg.eval.eval_every = 100
    cfg.eval.eval_games = 20

if PIPELINE == 'exit_blend':
    # Phase 3 (stronger expert search): grounded learned value at search leaves,
    # BLENDED with the heuristic (NOT swapped -- a pure swap collapsed 77%->0%).
    # Each candidate leaf is scored by both evaluate_state and the value head,
    # z-scored across the decision's leaves, then combined as
    #   score = (1 - w)*z(heur) + w*z(neural).
    # Warm-started from the iter-20 ckpt. Run once per w in {0,0.3,0.5,0.7}; each
    # weight gets its own run_name so the A/B never collides on Drive.
    w = float(VALUE_LEAF_BLEND)
    cfg.exit.value_leaf_blend = w
    cfg.exit.neural_value_leaves = False   # blend path, not the parked pure-swap
    cfg.run_name = f'v2_exit_blend_w{int(round(w*100)):03d}_a100'
    WARM_START = f'{DRIVE_OUTPUT}/checkpoints/v2_exit_a100/ckpt_000020.pt'

if PIPELINE == 'exit_neural':
    # Phase 5 (PARKED): neural-value ExIt -- a pure heuristic->neural SWAP at
    # leaves. Collapsed the 77% agent (sibling ranking uncorrelated). Kept for
    # reference; prefer exit_blend. Warm-started from iter-20. neural_value_leaves
    # from the yaml.
    cfg.run_name = 'v2_exit_neural_a100'
    cfg.exit.neural_value_leaves = True
    cfg.exit.value_leaf_blend = 0.0
    WARM_START = f'{DRIVE_OUTPUT}/checkpoints/v2_exit_a100/ckpt_000020.pt'

if PIPELINE == 'exit_rollout':
    # Phase 1 (CONFIRMED NEGATIVE): every-step in-sim ROLLOUT opponent. Regressed
    # the 77% agent (weak hand-coded rollout poisons leaf values). Kept for
    # reference. Warm-started from the iter-20 ckpt.
    cfg.run_name = 'v2_exit_rollout_a100'
    cfg.exit.rollout_search = True       # the experiment
    cfg.exit.rollout_self = True         # symmetric: our continuation rolls out too
    cfg.exit.two_player_search = False   # keep the failed sparse turn-1 version OFF
    WARM_START = f'{DRIVE_OUTPUT}/checkpoints/v2_exit_a100/ckpt_000020.pt'

if PIPELINE == 'gumbel':
    # Phase 5 (stronger expert search) -- Gumbel/Sequential-Halving candidate
    # selection (Build 1) anchored to the policy prior, optionally with the
    # strong-net in-sim opponent (Build 2). Replaces the refuted value-blend and
    # the weak rollout opponent. Heuristic leaf + passive sim by default; the
    # gumbel_sims/top_m/c_visit knobs come from configs/v2_exit_gumbel.yaml.
    # Warm-started from the iter-20 ckpt. Run the control (GUMBEL_SEARCH=False)
    # and the experiment as SEPARATE runs so the A/B never collides on Drive.
    cfg.exit.gumbel_search = bool(GUMBEL_SEARCH)
    cfg.exit.net_opponent = bool(GUMBEL_SEARCH and NET_OPPONENT)
    cfg.exit.gumbel_c_scale = float(GUMBEL_C_SCALE)
    cfg.exit.value_leaf_blend = 0.0       # not the refuted blend
    cfg.exit.neural_value_leaves = False  # not the parked pure-swap
    _tag = ('netopp' if cfg.exit.net_opponent
            else ('on' if cfg.exit.gumbel_search else 'ctrl'))
    cfg.run_name = f'v2_exit_gumbel_{_tag}_a100'
    WARM_START = f'{DRIVE_OUTPUT}/checkpoints/v2_exit_a100/ckpt_000020.pt'

print(f'PIPELINE      : {PIPELINE}')
print(f'run_name      : {cfg.run_name}')
print(f'device        : {cfg.device}')
print(f'model         : embed={cfg.model.embed_dim} layers={cfg.model.n_layers} '
      f'n_fractions={cfg.model.n_fractions}')
print(f'ship_fractions: {cfg.env.ship_fractions}')
print(f'BC            : enabled={cfg.imitation.enabled} games={cfg.imitation.bc_games} '
      f'epochs={cfg.imitation.bc_epochs} cache={cfg.imitation.bc_cache_path}')
if PIPELINE in EXIT_FAMILY:
    print(f'ExIt          : iters={cfg.exit.iterations} games/iter={cfg.exit.games_per_iter} '
          f'depth={cfg.exit.search_depth} cand={cfg.exit.search_candidates} '
          f'workers={cfg.exit.search_workers} batch={cfg.exit.train_batch_size}')
    print(f'              : collect_workers={cfg.exit.collect_workers} fast_env={cfg.exit.collect_fast_env}')
    print(f'eval (gate)   : games={cfg.eval.eval_games} seed={cfg.eval.eval_seed} '
          f'workers={cfg.eval.eval_workers} (side-alternated, paired)')
else:
    print(f'PPO           : updates={cfg.ppo.total_updates} envs={cfg.ppo.num_envs} '
          f'workers={cfg.ppo.num_workers} rollout={cfg.ppo.rollout_steps} '
          f'minibatch={cfg.ppo.minibatch_size} gamma={cfg.ppo.gamma} '
          f'value_symlog={cfg.ppo.value_symlog} ent={cfg.ppo.ent_coef}->{cfg.ppo.ent_coef_end}')
if PIPELINE in ('v3', 'v4'):
    print(f'feats         : pair={cfg.env.use_pair_features}(dim={cfg.env.pair_feat_dim}) '
          f'comet_targeting={cfg.env.comet_targeting} planet_feat_dim={cfg.model.planet_feat_dim}')
    print(f'reward        : {cfg.reward.reward_mode} | PFSP: enabled={cfg.pfsp_enabled} '
          f'apex_floor={cfg.pfsp_apex_min_prob}')
if PIPELINE == 'v4':
    print(f'v4 ceiling    : popart={cfg.ppo.popart} aux_value={cfg.model.aux_value_head} '
          f'shot_head={cfg.model.shot_success_head} coef_floor={cfg.imitation.coef_floor}')
    print(f'              : rich_global={cfg.env.rich_global_features} timeline={cfg.env.timeline_features} '
          f'canonical_rot={cfg.env.canonical_rotation} req_frac={cfg.env.requirement_relative_fractions}')
if PIPELINE == 'exit_blend':
    print(f'Phase 3 blend : value_leaf_blend={cfg.exit.value_leaf_blend}  '
          f'(0=heuristic control)  warm_start={WARM_START}')
if PIPELINE == 'exit_neural':
    print(f'Phase 5       : neural_value_leaves={cfg.exit.neural_value_leaves}  warm_start={WARM_START}')
if PIPELINE == 'exit_rollout':
    print(f'Phase 1       : rollout_search={cfg.exit.rollout_search} '
          f'rollout_self={cfg.exit.rollout_self} two_player={cfg.exit.two_player_search}  '
          f'warm_start={WARM_START}')
if PIPELINE == 'gumbel':
    print(f'Phase 5 build : gumbel_search={cfg.exit.gumbel_search} '
          f'(sims={cfg.exit.gumbel_sims} top_m={cfg.exit.gumbel_top_m} '
          f'c_scale={cfg.exit.gumbel_c_scale})  net_opponent={cfg.exit.net_opponent}')
    print(f'              : run_name={cfg.run_name}  warm_start={WARM_START}')


In [ ]:
#@title 4. Train

import os, yaml
from v2.config import v2_config_to_dict

temp_cfg = '/tmp/v2_train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
print(f'Wrote config -> {temp_cfg}')

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'

# Resume: continue from this run's ckpt_last.pt on Drive. The train script then
# SKIPS BC pretrain/demo collection, restores model+optimizer, advances the RNG
# seed past completed updates, and reloads cached demos if the imitation coef is
# still > 0 at the resume point.
RESUME = True  #@param {type:"boolean"}
resume_ckpt = f'{CHECKPOINT_DIR}/ckpt_last.pt'
resume_flag = ''
if RESUME and os.path.exists(resume_ckpt):
    resume_flag = f'--resume "{resume_ckpt}"'           # continue this run
    print(f'Resuming from {resume_ckpt}')
elif WARM_START and os.path.exists(WARM_START):
    resume_flag = f'--resume "{WARM_START}"'            # Phase 5 warm-start (skips BC)
    print(f'Warm-starting from {WARM_START}')
elif RESUME:
    print(f'RESUME=True but no checkpoint at {resume_ckpt}; starting fresh.')

module = 'v2.exit_train' if PIPELINE in EXIT_FAMILY else 'v2.train'
print(f'Launching: python -m {module} --config {temp_cfg} {resume_flag}\n')
!python -m {module} --config {temp_cfg} {resume_flag}

print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Submission

Builds a **trained-model bundle** dir *and* packs it into the **`submission.tar.gz`**
you actually upload: `main.py` + `ckpt_last.pt` + `submission_config.yaml` + the
`v2`/`src` packages. At match time Kaggle extracts the archive into
`/kaggle_simulations/agent/` and execs `main.py`, which auto-loads the checkpoint
and config from there.

⚠️ **Submit the `.tar.gz`, not the bundle directory, and `main.py`/`v2`/`src` must
sit at the *archive root*** (siblings of each other). If `v2/` ends up nested under
a bundle-dir prefix — or omitted entirely (e.g. uploading only `main.py`) — the
agent crashes with `ModuleNotFoundError: No module named 'v2'`. The cell below
builds the archive with `arcname` at root and self-checks it by extracting +
running one step before you upload.

- **v3** uses `v2/agent_v3.py` (the REAL feature pipeline), so the submitted
  policy sees byte-identical pair/comet features to training. The bundled
  `submission_config.yaml` tells it which feature flags + model dims to build.
- **ppo/exit** use the lighter inlined `v2/agent.py`.
- An **apex** baseline is also saved as a reliable fallback.

In [ ]:
#@title 5. Generate Submission

import shutil, yaml, tarfile, os, sys, tempfile
from pathlib import Path
from v2.config import v2_config_to_dict

sub_dir = Path(DRIVE_OUTPUT) / 'submissions'
sub_dir.mkdir(parents=True, exist_ok=True)

# Reliable apex baseline
try:
    from agents.rl_agent import export_submission
    export_submission(None, output_path=str(sub_dir / 'submission_apex.py'), mode='apex')
    print(f'Apex baseline: {sub_dir / "submission_apex.py"}')
except Exception as e:
    print(f'(apex export skipped: {e})')

# Trained-model bundle
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
bundle = sub_dir / f'{cfg.run_name}_bundle'
tarball = sub_dir / f'{cfg.run_name}_submission.tar.gz'
if ckpt_src.exists():
    bundle.mkdir(parents=True, exist_ok=True)
    # v3/v4 need the real feature pipeline + their config; v2 uses the inlined agent.
    agent_src = 'v2/agent_v3.py' if PIPELINE in ('v3', 'v4') else 'v2/agent.py'
    shutil.copy2(agent_src, bundle / 'main.py')
    shutil.copy2(ckpt_src, bundle / 'ckpt_last.pt')
    shutil.copytree('v2', bundle / 'v2', dirs_exist_ok=True)
    shutil.copytree('src', bundle / 'src', dirs_exist_ok=True)
    # Bundle the exact config so the agent rebuilds matching features/model.
    with open(bundle / 'submission_config.yaml', 'w') as f:
        yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
    print(f'{PIPELINE} bundle: {bundle}')
    print('  contents: main.py + ckpt_last.pt + submission_config.yaml + v2/ + src/')

    # --- Build the actual submission.tar.gz ---------------------------------
    # CRITICAL: main.py + v2/ + src/ must sit at the ARCHIVE ROOT (arcname=name),
    # NOT nested under a bundle-dir prefix. Kaggle extracts the archive into
    # /kaggle_simulations/agent/, then execs /kaggle_simulations/agent/main.py.
    # If v2/ isn't a sibling of main.py there, main.py raises
    #   ModuleNotFoundError: No module named 'v2'
    ROOT_ENTRIES = ['main.py', 'ckpt_last.pt', 'submission_config.yaml', 'v2', 'src']
    if tarball.exists():
        tarball.unlink()
    with tarfile.open(tarball, 'w:gz') as tf:
        for name in ROOT_ENTRIES:
            tf.add(bundle / name, arcname=name)  # arcname=name -> top of archive

    # --- Self-check: FAITHFULLY replicate Kaggle's agent loader -------------
    # kaggle_environments.agent.get_last_callable does, in order:
    #   sys.path.append(dirname(main.py)); exec(code, {}); sys.path.pop()
    # then calls the returned agent LATER (lazily, first step). A naive
    # importlib.exec_module check would NOT catch a regression here because it
    # sets __file__ and never pops the path. We mimic the real sequence so a
    # broken bootstrap (a bundle import deferred until AFTER the pop) fails
    # HERE, locally -- not silently in the competition.
    with tempfile.TemporaryDirectory() as td:
        with tarfile.open(tarball) as tf:
            tops = sorted({m.name.split('/')[0] for m in tf.getmembers()})
            tf.extractall(td)
        assert tops == sorted(ROOT_ENTRIES), f'archive root wrong: {tops}'
        assert (Path(td) / 'v2' / '__init__.py').exists(), 'v2 not a package at root'
        main_path = str(Path(td) / 'main.py')
        cwd0 = os.getcwd()
        # Cold import: drop any cached bundle modules so resolution is real.
        for _m in [k for k in list(sys.modules)
                   if k in ('v2', 'src') or k.startswith(('v2.', 'src.'))]:
            del sys.modules[_m]
        try:
            os.chdir(td)                       # Kaggle: agent dir is cwd
            env = {}                           # Kaggle: bare namespace, NO __file__
            code = compile(open(main_path).read(), main_path, 'exec')
            sys.path.append(td)                # Kaggle: append exec_dir
            exec(code, env)                    # bundle imports must resolve HERE
            sys.path.pop()                     # Kaggle: pop right after exec
            agent_fn = [v for v in env.values() if callable(v)][-1]
            assert getattr(agent_fn, '__name__', None) == 'agent', 'agent must be last callable'
            from kaggle_environments import make
            obs = make('orbit_wars').reset(num_agents=2)[0]['observation']
            mv = agent_fn(obs)                 # lazy first call -- used to crash post-pop
        finally:
            os.chdir(cwd0)
    sz = tarball.stat().st_size / 1e6
    print(f'\nSUBMISSION TARBALL OK -> {tarball} ({sz:.1f} MB)')
    print(f'  archive root: {tops}')
    print(f'  self-check (Kaggle exec->pop semantics): agent() ran -> {len(mv)} move(s)')
else:
    print(f'No checkpoint at {ckpt_src} yet.')

print('\nUpload:')
print(f'  kaggle competitions submit orbit-wars -f "{sub_dir/"submission_apex.py"}" -m "apex"')
if ckpt_src.exists():
    print(f'  kaggle competitions submit orbit-wars -f "{tarball}" -m "{cfg.run_name}"')
    print('  (submit the .tar.gz, NOT the bundle dir -- files must be at archive root)')


## Evaluation

Run the trained OrbitNet head-to-head vs apex and random.

In [ ]:
#@title 6. Evaluate Trained Model

import torch
from pathlib import Path
from v2.model import OrbitNet
from v2.train import make_v2_eval_agent
from evaluation.evaluate import run_games, print_results
from agents.apex import agent as apex_agent
from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent

ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint at {ckpt_path}. Available:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = OrbitNet(cfg.model).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f'Loaded {ckpt_path} (iter/update {ckpt.get("update", "?")})')
    eval_agent = make_v2_eval_agent(model, cfg, device)

    N = 20
    print(f'\nRL vs Apex ({N} games):')
    print_results('v2', 'apex', run_games(eval_agent, apex_agent, n_games=N))
    print(f'RL vs Random ({N} games):')
    print_results('v2', 'random', run_games(eval_agent, random_agent, n_games=N))

## Phase 3 gate — `exit_blend` A/B (side-alternated)

Scores each blend weight's checkpoints with `scripts/eval_fast.py`'s engine
(`FastOrbitWars`, **side-alternated**, **paired seeds**) across **multiple seed
batches** — win-rate is high-variance across map seeds, so a single batch is not
enough. A blend weight ships only if it **beats the `w=0` control on the same
seeds**. Run this after training the weights you want to compare (each is a
separate Train run; `w=0.0` is the control).


In [ ]:
#@title Phase 3 gate: eval_blend A/B vs apex (side-alternated, multi-seed)
import os
from pathlib import Path
from v2.config import load_v2_config, v2_config_to_dict
from scripts.eval_fast import _eval_checkpoint

# Weights to compare (must already be trained — each is its own Train run).
BLEND_WEIGHTS = [0.0, 0.3, 0.5, 0.7]   #@param
ITER          = 'last'                 #@param   {type:"string"}  # 'last' or e.g. '20'
GAMES         = 60                      #@param   {type:"integer"} # per seed batch
SEED_BATCHES  = [20000, 31000, 42000]  #@param   # win-rate is seed-variant -> multiple
EVAL_WORKERS  = CPU_COUNT

# eval_fast reads ROOT/outputs/checkpoints/<run>; our runs live on Drive. Symlink
# the local dir at the Drive checkpoints so --run resolves without copying.
local_ckpt_root = Path('outputs/checkpoints')
local_ckpt_root.parent.mkdir(parents=True, exist_ok=True)
drive_ckpt_root = Path(f'{DRIVE_OUTPUT}/checkpoints')
if local_ckpt_root.is_symlink() or local_ckpt_root.exists():
    if not (local_ckpt_root.is_symlink() and local_ckpt_root.resolve() == drive_ckpt_root.resolve()):
        print(f'NOTE: {local_ckpt_root} already exists and is not the Drive symlink; '
              f'eval_fast reads it as-is.')
else:
    local_ckpt_root.symlink_to(drive_ckpt_root, target_is_directory=True)
    print(f'symlinked {local_ckpt_root} -> {drive_ckpt_root}')

cfg_dict = v2_config_to_dict(load_v2_config('configs/v2_exit_neural.yaml'))
ckpt_name = 'ckpt_last.pt' if ITER == 'last' else f'ckpt_{int(ITER):06d}.pt'

def run_name_for(w):
    return f'v2_exit_blend_w{int(round(float(w)*100)):03d}_a100'

print(f'iter={ITER}  games/seed={GAMES}  seeds={SEED_BATCHES}  workers={EVAL_WORKERS}\n')
header = f'{"w":>5} | ' + ' '.join(f'seed{s:>6}' for s in SEED_BATCHES) + f' | {"mean":>5}'
print(header); print('-' * len(header))
results = {}
for w in BLEND_WEIGHTS:
    run = run_name_for(w)
    path = drive_ckpt_root / run / ckpt_name
    if not path.exists():
        print(f'{w:>5} | (no checkpoint at {run}/{ckpt_name} — train this weight first)')
        continue
    wrs = []
    for s in SEED_BATCHES:
        win, loss, tie = _eval_checkpoint(path, cfg_dict, GAMES, EVAL_WORKERS, 'apex', s)
        wrs.append(win / max(win + loss + tie, 1))
    results[w] = wrs
    mean = sum(wrs) / len(wrs)
    print(f'{w:>5.2f} | ' + ' '.join(f'{r:>9.0%}' for r in wrs) + f' | {mean:>5.0%}')

# Verdict: blend must beat the w=0 control on the SAME seeds.
if 0.0 in results:
    ctrl = results[0.0]
    print('\nvs control (w=0), per-seed win-rate delta:')
    for w, wrs in results.items():
        if w == 0.0:
            continue
        deltas = [a - b for a, b in zip(wrs, ctrl)]
        beats = all(d > 0 for d in deltas)
        print(f'  w={w:.2f}: ' + ' '.join(f'{d:+.0%}' for d in deltas)
              + f'   mean {sum(deltas)/len(deltas):+.1%}   '
              + ('SHIP (beats control on every seed)' if beats else 'not on every seed'))
else:
    print('\n(no w=0 control trained — train v2_exit_blend_w000_a100 to gate against.)')


## Phase 5 gate — `gumbel` (Build 1/2) vs control (side-alternated)

Each run is its own Train run: set `GUMBEL_SEARCH` / `NET_OPPONENT` in the config
cell, then Train. The **control** is `GUMBEL_SEARCH=False`
(`v2_exit_gumbel_ctrl_a100`). A variant ships only if it beats the control on the
**same** paired seeds. Gate Build 1 (`v2_exit_gumbel_on_a100`) before Build 2
(`v2_exit_gumbel_netopp_a100`).

In [ ]:
#@title Phase 5 gate: gumbel (Build 1/2) vs control (side-alternated, multi-seed)
import os
from pathlib import Path
from v2.config import load_v2_config, v2_config_to_dict
from scripts.eval_fast import _eval_checkpoint

CONTROL_RUN  = 'v2_exit_gumbel_ctrl_a100'   #@param {type:"string"}
VARIANT_RUNS = ['v2_exit_gumbel_on_a100', 'v2_exit_gumbel_netopp_a100']  #@param
ITER         = 'last'                  #@param {type:"string"}  # 'last' or e.g. '20'
GAMES        = 60                       #@param {type:"integer"} # per seed batch
SEED_BATCHES = [20000, 31000, 42000]   #@param   # win-rate is seed-variant -> multiple
EVAL_WORKERS = CPU_COUNT

# eval_fast reads ROOT/outputs/checkpoints/<run>; our runs live on Drive. Symlink
# the local dir at the Drive checkpoints so --run resolves without copying.
local_ckpt_root = Path('outputs/checkpoints')
local_ckpt_root.parent.mkdir(parents=True, exist_ok=True)
drive_ckpt_root = Path(f'{DRIVE_OUTPUT}/checkpoints')
if not (local_ckpt_root.is_symlink() or local_ckpt_root.exists()):
    local_ckpt_root.symlink_to(drive_ckpt_root, target_is_directory=True)
    print(f'symlinked {local_ckpt_root} -> {drive_ckpt_root}')

cfg_dict = v2_config_to_dict(load_v2_config('configs/v2_exit_gumbel.yaml'))
ckpt_name = 'ckpt_last.pt' if ITER == 'last' else f'ckpt_{int(ITER):06d}.pt'

def score(run):
    path = drive_ckpt_root / run / ckpt_name
    if not path.exists():
        return None
    wrs = []
    for s in SEED_BATCHES:
        win, loss, tie = _eval_checkpoint(path, cfg_dict, GAMES, EVAL_WORKERS, 'apex', s)
        wrs.append(win / max(win + loss + tie, 1))
    return wrs

print(f'iter={ITER}  games/seed={GAMES}  seeds={SEED_BATCHES}  workers={EVAL_WORKERS}\n')
header = f'{"run":>30} | ' + ' '.join(f'seed{s:>6}' for s in SEED_BATCHES) + f' | {"mean":>5}'
print(header); print('-' * len(header))
all_runs = [CONTROL_RUN] + [r for r in VARIANT_RUNS if r]
results = {}
for run in all_runs:
    wrs = score(run)
    if wrs is None:
        print(f'{run:>30} | (no checkpoint at {run}/{ckpt_name} — train it first)')
        continue
    results[run] = wrs
    print(f'{run:>30} | ' + ' '.join(f'{r:>9.0%}' for r in wrs) + f' | {sum(wrs)/len(wrs):>5.0%}')

# Verdict: each variant must beat the control on the SAME seeds.
if CONTROL_RUN in results:
    ctrl = results[CONTROL_RUN]
    print('\nvs control, per-seed win-rate delta:')
    for run, wrs in results.items():
        if run == CONTROL_RUN:
            continue
        deltas = [a - b for a, b in zip(wrs, ctrl)]
        beats = all(d > 0 for d in deltas)
        print(f'  {run}: ' + ' '.join(f'{d:+.0%}' for d in deltas)
              + f'   mean {sum(deltas)/len(deltas):+.1%}   '
              + ('SHIP (beats control on every seed)' if beats else 'not on every seed'))
else:
    print(f'\n(no control trained — train {CONTROL_RUN} (GUMBEL_SEARCH=False) first.)')

## Monitoring

Each run writes a resume-safe **`eval_history.csv`** to its Drive log dir
(appended + closed every eval, so the FUSE mount actually syncs it and the
full curve survives Colab session restarts). Plot it any time — even from a
second runtime while training continues — with the eval-curve cell below.
TensorBoard (last cell) shows train/*, eval/*, bc/* and can block cells below it.

In [ ]:
#@title 7. Plot eval win-rate curve (from eval_history.csv)

from pathlib import Path
from IPython.display import Image, display

csv_path = Path(CHECKPOINT_DIR.replace('/checkpoints/', '/logs/')) / 'eval_history.csv'
out_png = f'{DRIVE_OUTPUT}/logs/{cfg.run_name}_eval.png'
if csv_path.exists():
    !python scripts/plot_eval_curve.py --csv "{csv_path}" --out "{out_png}" --title "{cfg.run_name} — eval win rate"
    display(Image(out_png))
else:
    print(f'No eval_history.csv yet at {csv_path} (appears after the first eval).')

In [ ]:
#@title 8. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs